# Pipeline: k-mers

**Responsable:** Juan Arias

**Objetivo:** Calcular los perfiles de k-mers (k = 1 a 6) y cargarlos en `kmers` según MODELO.md.

Modelo de referencia: `../MODELO.md`

`DB_NAME` elige la base: `panteon_prueba` para probar y `panteon` para la carga real. La estrategia principal (MODELO.md) va a `DB_NAME.kmers`; la alternativa embebida va a `{DB_NAME}_bench.tes_embebidos`, para el benchmark. La colección `tes` original no se modifica.

In [1]:
import random
import sys
import time
from pathlib import Path

import pandas as pd
from pymongo import ASCENDING, MongoClient

sys.path.insert(0, "..")
from src.kmers import K_RANGE, all_kmer_counts, kmer_counts

DB_NAME = "panteon_prueba"
LOTE = 2000
RESULTADOS = Path("../resultados")
RESULTADOS.mkdir(exist_ok=True)

client = MongoClient("mongodb://localhost:27017")
db = client[DB_NAME]
db_bench = client[f"{DB_NAME}_bench"]
print(DB_NAME, "| TEs:", db.tes.count_documents({}), "| secuencias:", db.secuencias.count_documents({}))

panteon_prueba | TEs: 25 | secuencias: 25


## 1. Función de k-mers (probar con `ATGCA` del enunciado)

In [2]:
ejemplo = all_kmer_counts("ATGCA")
print(ejemplo[1])
print(ejemplo[2])
assert ejemplo[1]["counts"] == {"A": 2, "T": 1, "G": 1, "C": 1}
assert ejemplo[2]["counts"] == {"AT": 1, "TG": 1, "GC": 1, "CA": 1}

{'valid_windows': 5, 'counts': {'A': 2, 'C': 1, 'G': 1, 'T': 1}}
{'valid_windows': 4, 'counts': {'AT': 1, 'CA': 1, 'GC': 1, 'TG': 1}}


**Conclusión:** coincide con el enunciado. La implementación está en `src/kmers.py` (numpy + `np.bincount`) y sus tests en `tests/test_kmers.py`.

## 2. Tratamiento de N y otros caracteres

In [3]:
print(kmer_counts("ACGTNACGT", 3))
print(kmer_counts("acgtRacgt", 3))
print(all_kmer_counts("ACGTA")[6])

{'valid_windows': 4, 'counts': {'ACG': 2, 'CGT': 2}}
{'valid_windows': 4, 'counts': {'ACG': 2, 'CGT': 2}}
{'valid_windows': 0, 'counts': {}}


**Conclusión:** se descarta toda ventana con un carácter distinto de A, C, G, T (N o IUPAC); `valid_windows` cuenta solo las ventanas usadas, así que las frecuencias son `counts / valid_windows`. Se pasa a mayúsculas antes de contar. Si la secuencia es más corta que k el documento se guarda igual, con `valid_windows = 0` y `counts = {}`, para que cada TE tenga siempre 6 documentos.

## 3. Validación `$jsonSchema` de `kmers`

In [4]:
ESQUEMA_KMERS = {
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["te_id", "pdb_id", "te_order", "k", "valid_windows", "counts"],
        "properties": {
            "te_id": {"bsonType": "objectId"},
            "pdb_id": {"bsonType": "string"},
            "te_order": {"bsonType": "string"},
            "k": {"bsonType": "int", "minimum": 1, "maximum": 6},
            "valid_windows": {"bsonType": ["int", "long"], "minimum": 0},
            "counts": {"bsonType": "object"},
        },
    }
}

# Borra y recrea las dos colecciones: el notebook se puede ejecutar varias veces
db.drop_collection("kmers")
db.create_collection("kmers", validator=ESQUEMA_KMERS, validationLevel="strict", validationAction="error")
db_bench.drop_collection("tes_embebidos")
db_bench.create_collection("tes_embebidos")

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'panteon_prueba_bench'), 'tes_embebidos')

## 4. Procesamiento y carga por lotes

Cada estrategia recorre `tes` por lotes de 2.000, trae sus secuencias con `$in` y mide aparte el cálculo (k-mers y armado de documentos) y la inserción (`insert_many(ordered=False)`). La lectura desde MongoDB no entra en ninguno de los dos tiempos.

In [5]:
def lotes():
    """Lotes de (te, secuencia) leyendo tes y secuencias por _id."""
    cursor = db.tes.find({}).sort("_id", 1)
    while True:
        tes = [te for _, te in zip(range(LOTE), cursor)]
        if not tes:
            return
        seqs = {s["_id"]: s["sequence"] for s in db.secuencias.find({"_id": {"$in": [t["_id"] for t in tes]}})}
        yield [(te, seqs[te["_id"]]) for te in tes]


def docs_kmers(te, perfil):
    return [
        {"te_id": te["_id"], "pdb_id": te["pdb_id"], "te_order": te["classification"]["te_order"],
         "k": k, "valid_windows": p["valid_windows"], "counts": p["counts"]}
        for k, p in perfil.items()
    ]


def doc_embebido(te, perfil):
    return {**te, "kmers": {f"k{k}": p for k, p in perfil.items()}}


def cargar(coleccion, armar):
    n_tes = n_docs = 0
    t_calculo = t_insercion = 0.0
    for lote in lotes():
        t0 = time.perf_counter()
        docs = []
        for te, seq in lote:
            resultado = armar(te, all_kmer_counts(seq))
            docs.extend(resultado if isinstance(resultado, list) else [resultado])
        t1 = time.perf_counter()
        coleccion.insert_many(docs, ordered=False)
        t2 = time.perf_counter()
        t_calculo += t1 - t0
        t_insercion += t2 - t1
        n_tes += len(lote)
        n_docs += len(docs)
    return {"n_tes": n_tes, "n_docs": n_docs, "t_calculo_s": round(t_calculo, 3), "t_insercion_s": round(t_insercion, 3)}


def crear_indices(coleccion, indices):
    t0 = time.perf_counter()
    for claves, opciones in indices:
        coleccion.create_index(claves, **opciones)
    return round(time.perf_counter() - t0, 3)


benchmark = []
r = cargar(db.kmers, docs_kmers)
benchmark.append({"estrategia": "coleccion_kmers", "base": DB_NAME, **r})
r = cargar(db_bench.tes_embebidos, doc_embebido)
benchmark.append({"estrategia": "embebido_en_tes", "base": db_bench.name, **r})
pd.DataFrame(benchmark)

,estrategia,base,n_tes,n_docs,t_calculo_s,t_insercion_s
0,coleccion_kmers,panteon_prueba,25,150,0.005,0.004
1,embebido_en_tes,panteon_prueba_bench,25,25,0.005,0.006


## 5. Índices

Los índices se crean después de la carga. En `tes_embebidos` se replica el índice único de `pdb_id` de `tes`.

In [6]:
benchmark[0]["t_indices_s"] = crear_indices(db.kmers, [
    ([("te_id", ASCENDING), ("k", ASCENDING)], {"unique": True}),
    ([("pdb_id", ASCENDING), ("k", ASCENDING)], {}),
    ([("k", ASCENDING), ("te_order", ASCENDING)], {}),
])
benchmark[1]["t_indices_s"] = crear_indices(db_bench.tes_embebidos, [
    ([("pdb_id", ASCENDING)], {"unique": True}),
])

tabla = pd.DataFrame(benchmark)
tabla.to_csv(RESULTADOS / "benchmark_carga.csv", index=False)
print(sorted(db.kmers.index_information()))
tabla

['_id_', 'k_1_te_order_1', 'pdb_id_1_k_1', 'te_id_1_k_1']

,estrategia,base,n_tes,n_docs,t_calculo_s,t_insercion_s,t_indices_s
0,coleccion_kmers,panteon_prueba,25,150,0.005,0.004,0.086
1,embebido_en_tes,panteon_prueba_bench,25,25,0.005,0.006,0.036


## 6. Verificación

In [7]:
n_tes = db.tes.count_documents({})
con_seis = db.kmers.aggregate([{"$group": {"_id": "$te_id", "n": {"$sum": 1}}}, {"$match": {"n": 6}}, {"$count": "n"}])
con_seis = next(con_seis, {"n": 0})["n"]
print(f"TEs con exactamente 6 documentos: {con_seis} de {n_tes}")
assert con_seis == n_tes
assert db.kmers.count_documents({}) == 6 * n_tes
assert db_bench.tes_embebidos.count_documents({}) == n_tes

# 5 TEs al azar: el perfil recalculado coincide con el guardado en ambas estrategias
muestra = random.Random(0).sample(db.tes.distinct("_id"), 5)
for te_id in muestra:
    esperado = all_kmer_counts(db.secuencias.find_one({"_id": te_id})["sequence"])
    en_kmers = {d["k"]: {"valid_windows": d["valid_windows"], "counts": d["counts"]}
                for d in db.kmers.find({"te_id": te_id})}
    embebido = db_bench.tes_embebidos.find_one({"_id": te_id})["kmers"]
    en_embebido = {int(k[1:]): v for k, v in embebido.items()}
    assert en_kmers == esperado and en_embebido == esperado, te_id
print("Perfiles de 5 TEs al azar: coinciden en ambas estrategias")

# tes original intacta
assert "kmers" not in db.tes.find_one()

TEs con exactamente 6 documentos: 25 de 25
Perfiles de 5 TEs al azar: coinciden en ambas estrategias
